# QuantAI specialist model — training and evaluation

This notebook fine-tunes an open-weight instruction model for QuantAI using LoRA/QLoRA. It does **not** call a paid LLM chat API. The resulting model still depends on its chosen pretrained base; this is specialised fine-tuning, not pretraining from scratch.

**Recommended:** choose a GPU runtime before starting. Training can fail or run very slowly without a GPU. Free GPU availability and session duration vary. The starter dataset is a pipeline smoke-test dataset, not a publication-quality corpus; review and expand it before treating results as reliable.

No Hugging Face access token is needed to download a public base model or train locally. Do not add secrets or private research data to this notebook.

In [ ]:
import os, subprocess, pathlib, torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // (1024**3), 'GiB')
else: print('Select Runtime > Change runtime type > GPU before training.')
REPO = '/content/model-analysis--hub'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','--branch','quantai-local-specialist-v1','https://github.com/kamaldeenjnr/model-analysis--hub.git',REPO], check=True)
os.chdir(os.path.join(REPO,'quantai-model'))
print('Working directory:', os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

## 1. Validate the data

This validates file structure, supported actions, minimum counts and exact prompt separation. It does not establish that the examples are scientifically correct. Review training and evaluation examples before a serious run.

In [ ]:
!python validate_data.py --train data/train.jsonl --eval data/eval.jsonl
!pytest -q test_local_service.py

## 2. Train the QuantAI adapter

This uses the Qwen3 4B Instruct base and LoRA/4-bit loading when the environment supports it. This is a practical starting point for limited compute, not a claim that a small model has expert-level performance. If the GPU runs out of memory, reduce `--max-length` and batch settings or use a smaller base. Keep the held-out evaluation file unchanged during training.

In [ ]:
assert torch.cuda.is_available(), 'A GPU runtime is strongly recommended; stop here rather than unknowingly running a very slow CPU training job.'
!python train_lora.py --train data/train.jsonl --eval data/eval.jsonl --output outputs/quantai-qwen3-4b-lora --epochs 3 --max-length 1024

## 3. Evaluate against the untouched base model

Review the JSON reports and individual failures. Aggregate scores on a small synthetic set are only a smoke test. Do not publish performance claims from these fixtures.

In [ ]:
!python evaluate.py --model outputs/quantai-qwen3-4b-lora --cases data/eval.jsonl --report outputs/fine-tuned-evaluation.json
!python evaluate.py --model outputs/quantai-qwen3-4b-lora --baseline-only --cases data/eval.jsonl --report outputs/base-evaluation.json
import json
for p in ['outputs/base-evaluation.json','outputs/fine-tuned-evaluation.json']:
    d=json.load(open(p)); print('\n',p); print({k:v for k,v in d.items() if k not in ('results','warning')})

## 4. Export a standalone model

The merged model includes the base weights and trained adapter. It can run locally without a per-request inference API, but it requires enough RAM/VRAM and storage. Check base-model licensing before redistributing weights.

In [ ]:
!python export_model.py --adapter outputs/quantai-qwen3-4b-lora --output outputs/quantai-merged-model
print('Export exists:', pathlib.Path('outputs/quantai-merged-model').exists())

## 5. Optional: persist output to Google Drive

Mount Drive only if you want to keep the model after the notebook session. The merged model may take several gigabytes. Do not share a Drive folder containing private data or credentials.

In [ ]:
# Uncomment and run when you want to save the output to your own Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# target='/content/drive/MyDrive/QuantAI/quantai-merged-model'
# os.makedirs(os.path.dirname(target), exist_ok=True)
# shutil.copytree('outputs/quantai-merged-model', target, dirs_exist_ok=True)
# print('Saved to', target)

## Release checklist

- [ ] Review all training targets and the full held-out evaluation set.
- [ ] Compare base and fine-tuned failures by category.
- [ ] Expand the dataset with licensed, reviewed material and expert-authored examples.
- [ ] Test hallucination, prompt injection, missing variables, complex survey design and unsupported requests.
- [ ] Verify the local server on the target hardware and benchmark latency/memory.
- [ ] Keep statistical calculations in the deterministic engine and validate every requested action.
- [ ] Do not claim production readiness until independent review and acceptance thresholds pass.